# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Szan-12345/FLYRANK-MACHINE-LEARNING/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Connect to the warehouse
Same Hugging Face + DuckDB connection as previous weeks — reused unchanged so this
notebook runs independently.

In [2]:
# This cell is for CODE (numbers, a query, a check).
%pip -q install duckdb huggingface_hub

import os, getpass
import duckdb
import pandas as pd
import numpy as np
import json

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print('connected')

connected


## Week-4 rule constants (unchanged)

In [3]:
# This cell is for CODE (numbers, a query, a check).
WINDOW_DAYS       = 15
MIN_PREV_IMPR     = 50
IMPR_DROP_THRESH  = 0.20
CLICK_DROP_THRESH = 0.20
POS_SLIP_THRESH   = 1.0
ACTIVITY_DROP_FRAC= 0.25

W_IMPR, W_CLICK, W_POS, W_ACTIVITY = 0.40, 0.25, 0.20, 0.15
assert abs((W_IMPR + W_CLICK + W_POS + W_ACTIVITY) - 1.0) < 1e-9

REASON_CODES = ['IMPR_DROP', 'CLICK_DROP', 'POSITION_SLIP', 'ACTIVITY_DROP']
print('rule constants set')

rule constants set


## Rebuild the scored population and the Week-5/6 model
Same windowing query and target construction as previous weeks, plus the honest
client-grouped Logistic Regression from Week 6 — trained once here so this notebook
can produce a final blended queue.

In [5]:
# This cell is for CODE (numbers, a query, a check).
windowed = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS decision_day FROM {TABLES['fact_daily']}
    ),
    per_item AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            SUM(CASE WHEN f.report_date >  b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_impressions ELSE 0 END)                              AS imp_last,
            SUM(CASE WHEN f.report_date <= b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_impressions ELSE 0 END)                              AS imp_prev,
            SUM(CASE WHEN f.report_date >  b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_clicks ELSE 0 END)                                   AS clk_last,
            SUM(CASE WHEN f.report_date <= b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_clicks ELSE 0 END)                                   AS clk_prev,
            AVG(CASE WHEN f.report_date >  b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_avg_position END)                                    AS pos_last,
            AVG(CASE WHEN f.report_date <= b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_avg_position END)                                    AS pos_prev,
            COUNT(DISTINCT CASE WHEN f.report_date >  b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     AND f.gsc_impressions > 0 THEN f.report_date END)               AS active_days_last,
            COUNT(DISTINCT CASE WHEN f.report_date <= b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     AND f.gsc_impressions > 0 THEN f.report_date END)               AS active_days_prev,
            MAX(b.decision_day)                                                      AS decision_day
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.decision_day - INTERVAL ({2*WINDOW_DAYS}) DAY
          AND f.gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT * FROM per_item
    WHERE imp_prev >= {MIN_PREV_IMPR}
""").df()

d = windowed.copy()
d['pct_impr_change']  = (d['imp_last'] - d['imp_prev']) / d['imp_prev']
d['pct_click_change'] = np.where(d['clk_prev'] > 0, (d['clk_last'] - d['clk_prev']) / d['clk_prev'], np.nan)
d['pos_change']       = d['pos_last'] - d['pos_prev']
d['activity_change']  = (d['active_days_prev'] - d['active_days_last']) / d['active_days_prev'].clip(lower=1)

d['impr_drop_score']     = (-d['pct_impr_change']).clip(lower=0, upper=1)
d['click_drop_score']    = (-d['pct_click_change']).clip(lower=0, upper=1).fillna(0)
d['pos_slip_score']      = (d['pos_change'] / 5.0).clip(lower=0, upper=1)
d['activity_drop_score'] = d['activity_change'].clip(lower=0, upper=1)

def reasons(row):
    fired = []
    if row['pct_impr_change'] <= -IMPR_DROP_THRESH: fired.append('IMPR_DROP')
    if pd.notna(row['pct_click_change']) and row['pct_click_change'] <= -CLICK_DROP_THRESH: fired.append('CLICK_DROP')
    if row['pos_change'] >= POS_SLIP_THRESH: fired.append('POSITION_SLIP')
    if row['activity_change'] >= ACTIVITY_DROP_FRAC: fired.append('ACTIVITY_DROP')
    return fired

d['reason_codes']    = d.apply(reasons, axis=1)
d['n_reasons']        = d['reason_codes'].apply(len)
d['reason_codes_str'] = d['reason_codes'].apply(lambda r: '|'.join(r) if r else 'NONE')

FEATURES = ['pct_impr_change', 'pct_click_change', 'pos_change', 'activity_change', 'log_imp_prev']

d2 = d.copy()
d2['high_confidence_decline'] = (
    (d2['n_reasons'] > 0) & (d2['imp_prev'] >= MIN_PREV_IMPR * 5)
).astype(int)
d2['log_imp_prev'] = np.log1p(d2['imp_prev'])
d2 = d2.dropna(subset=FEATURES + ['high_confidence_decline']).reset_index(drop=True)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, te_idx = next(gss.split(d2, groups=d2['client_hash_id']))
train_grp = d2.iloc[tr_idx]

clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
clf.fit(train_grp[FEATURES], train_grp['high_confidence_decline'])

d2['model_score'] = clf.predict_proba(d2[FEATURES])[:, 1]
print(f'{len(d2):,} items scored | model trained on {len(train_grp):,} client-grouped rows')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

58,667 items scored | model trained on 44,038 client-grouped rows


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*



**The queue:** rank every flagged item by a blended score — the rule's `action_score`
(Week 4) as the primary sort, with the Week-5/6 model's `model_score` shown alongside
as a secondary confidence signal, not a replacement. Week 6 found the model's own
top-ranked picks don't strictly dominate the rule's (precision@20 tension observed in
Week 5), so this queue leads with the transparent rule and uses the model to flag which
rule-picks look most trustworthy — not to override the rule outright.

**Why blended, not model-only:** the model's `log_imp_prev` reliance (Week 5 Section 4)
means it's very good at spotting *volume-backed* declines but weaker on narrower cases
(e.g. rising-impressions-but-falling-CTR) that the rule's `CLICK_DROP` code catches on
its own. A human reviewer benefits from both signals, in plain words, side by side.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
queue = d2[d2['n_reasons'] > 0].copy()
queue['action_score'] = (
    W_IMPR     * queue['impr_drop_score'] +
    W_CLICK    * queue['click_drop_score'] +
    W_POS      * queue['pos_slip_score'] +
    W_ACTIVITY * queue['activity_drop_score']
).round(4)

queue['model_confidence'] = pd.cut(
    queue['model_score'], bins=[0, 0.33, 0.66, 1.0],
    labels=['low', 'medium', 'high'], include_lowest=True
)

queue = queue.sort_values('action_score', ascending=False).reset_index(drop=True)
queue.insert(0, 'rank', queue.index + 1)

# Map each reason code to a plain-language recommended action
ACTION_MAP = {
    'IMPR_DROP':     'Check indexing status; confirm the page is still crawlable and ranking',
    'CLICK_DROP':    'Review title/meta description — likely a SERP snippet or CTR issue',
    'POSITION_SLIP': 'Audit for a ranking-specific cause (algorithm update, competitor, relevance)',
    'ACTIVITY_DROP': 'Highest urgency — check for deindexing or a technical/crawl failure first',
}
def recommend(codes):
    if not codes:
        return 'No action'
    # order by urgency: ACTIVITY_DROP > POSITION_SLIP > IMPR_DROP > CLICK_DROP
    priority = ['ACTIVITY_DROP', 'POSITION_SLIP', 'IMPR_DROP', 'CLICK_DROP']
    lead = next((c for c in priority if c in codes), codes[0])
    return ACTION_MAP[lead]

queue['recommended_action'] = queue['reason_codes'].apply(recommend)

print(f"{len(queue):,} items in the ranked queue")
queue[['rank','action_score','model_score','model_confidence','reason_codes_str','recommended_action']].head(15)


47,332 items in the ranked queue


,rank,action_score,model_score,model_confidence,reason_codes_str,recommended_action
0,1,0.9882,0.529265,medium,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,Highest urgency — check for deindexing or a te...
1,2,0.9839,0.292660,low,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,Highest urgency — check for deindexing or a te...
2,3,0.9820,0.242998,low,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,Highest urgency — check for deindexing or a te...
3,4,0.9816,0.234140,low,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,Highest urgency — check for deindexing or a te...
4,5,0.9809,0.234925,low,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,Highest urgency — check for deindexing or a te...
5,6,0.9803,0.325393,low,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,Highest urgency — check for deindexing or a te...
6,7,0.9784,0.937981,high,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,Highest urgency — check for deindexing or a te...
7,8,0.9755,0.657517,medium,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,Highest urgency — check for deindexing or a te...
8,9,0.9744,0.439341,medium,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,Highest urgency — check for deindexing or a te...
9,10,0.9708,0.271558,low,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP,Highest urgency — check for deindexing or a te...


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*


**Who uses this:** a content strategist or SEO reviewer doing weekly/monthly triage —
someone deciding which of hundreds of declining pages to look at *first*, not someone
looking for a final verdict on any single page.

**What it's for:** prioritization only. The queue orders review effort; it does not
diagnose the cause of a decline, and it does not confirm that a decline is real vs. a
seasonal or platform-wide blip.

**Where it stops being valid:**
- **New clients / cold start.** The model was trained on this portfolio's clients
  (Week-6 grouped split). A brand-new client with no prior-window history isn't covered
  by `imp_prev >= MIN_PREV_IMPR` and won't appear in the queue at all — this is a
  coverage gap, not a "this client is healthy" signal.
- **Proxy target, not ground truth.** `high_confidence_decline` (Week 5) is a label we
  built from our own judgment call, not a confirmed outcome like verified traffic
  recovery. Treat model_confidence as directional support, not a certified score.
- **Site-wide events aren't separated from page-level ones.** A migration or Core Update
  hitting one client will flood that client's items to the top of the queue together
  (Week 4's own top-20 review found 45% concentration from one client) — the queue
  doesn't yet distinguish "many pages independently declining" from "one event, many
  symptoms."
- **Refresh-effect framing stays cautious.** The FlyRank paper's own refresh findings
  (Finding #4/#8) are suggestive external context for *why* refreshing might help, but
  our queue does not itself measure refresh outcomes — it only flags decline, it does
  not validate that refreshing fixes it. Any refresh-timing recommendation here is
  informed by the paper's directional finding, not proven on this portfolio's own data.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Quantify one of the stated limits directly: how much of the queue's top 20 comes from
# a single client, on this run — same check Week 4 did by hand, reproduced here.
client_share_top20 = queue.head(20)['client_hash_id'].value_counts(normalize=True).round(2)
print('client share of top 20 rows (site-wide-event risk check):')
print(client_share_top20)

# Coverage check: how many total distinct content items exist in fact_daily vs. how many
# clear the imp_prev floor and are therefore even eligible to appear in this queue
total_content = con.sql(f"SELECT COUNT(DISTINCT content_hash_id) AS n FROM {TABLES['fact_daily']}").df()['n'][0]
print(f"\nTotal distinct content items in fact_daily: {total_content:,}")
print(f"Items eligible for scoring (cleared imp_prev floor): {len(d):,}")
print(f"Coverage: {len(d)/total_content:.1%} of all content is even eligible for this queue")

client share of top 20 rows (site-wide-event risk check):
client_hash_id
client_62f4a7e64f5e0096    0.45
client_08a6a72ff48e62c0    0.25
client_23a62021009f63c4    0.10
client_73cda7b4e4f265ea    0.10
client_e00b29e582949543    0.05
client_0b245132bb722950    0.05
Name: proportion, dtype: float64


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Total distinct content items in fact_daily: 427,292
Items eligible for scoring (cleared imp_prev floor): 99,893
Coverage: 23.4% of all content is even eligible for this queue


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*



**What a person must check before acting on any queue item:**
1. Confirm the decline isn't explained by a known, already-communicated event
   (site migration, seasonal category, planned content retirement).
2. For `ACTIVITY_DROP` items specifically, check Search Console coverage/indexing status
   directly — this is the one reason code that can mean "the page technically broke,"
   which needs a developer, not a content edit.
3. Sanity-check `imp_prev` against the reliability floor discussion from Week 4 — items
   just above `MIN_PREV_IMPR * 1` carry more noise than items well above it; treat
   `model_confidence: low` items with proportionally more skepticism.
4. Cross-check `model_confidence` against `reason_codes` — if the rule and model
   disagree sharply (e.g. high action_score but low model_confidence), that's a case
   worth a closer look, not an automatic dismissal of either signal.

**What should NEVER be automated from this notebook's output:**
- **Auto-publishing or auto-editing content.** This queue identifies *candidates* for
  review; it has no signal about what a correct rewrite looks like.
- **Auto-unpublishing/deindexing pages.** A false positive here (e.g. a low-volume,
  noisy `IMPR_DROP`) could remove a page that was never actually failing.
- **Using this score as an employee performance metric.** The queue reflects a proxy
  target built on a specific rule threshold, not a validated measure of anyone's content
  quality or effort.
- **Client-facing reporting without review.** Given the site-wide-event concentration
  risk (Section 2), sending an unreviewed top-20 list to a client could overstate how
  many pages are independently, meaningfully declining.
- **Triggering budget or resourcing decisions directly off `action_score`.** The score
  is a triage ordering, not a value/impact estimate (see Section 4's cost/value note).

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Surface the specific cases most likely to need the extra scrutiny named above:
# rule/model disagreement, and low-confidence-but-high-rank items.
disagreement = queue[
    (queue['action_score'] > queue['action_score'].median()) &
    (queue['model_confidence'] == 'low')
]
print(f"{len(disagreement):,} items: high rule score but low model confidence — flag for extra review")
disagreement[['rank','action_score','model_score','reason_codes_str']].head(10)

3,584 items: high rule score but low model confidence — flag for extra review


,rank,action_score,model_score,reason_codes_str
1,2,0.9839,0.292660,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
2,3,0.9820,0.242998,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
3,4,0.9816,0.234140,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
4,5,0.9809,0.234925,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
5,6,0.9803,0.325393,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
9,10,0.9708,0.271558,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
10,11,0.9702,0.322973,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
11,12,0.9702,0.238277,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
12,13,0.9687,0.283339,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP
14,15,0.9678,0.242236,IMPR_DROP|CLICK_DROP|POSITION_SLIP|ACTIVITY_DROP


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

1. **Base-rate drift.** `high_confidence_decline`'s current rate (~42–59% depending on
   NaN handling, per Week 5/6) should be re-checked each run. A large swing (e.g. a
   Core Update affecting the whole portfolio at once) would mean the queue is temporarily
   dominated by one event rather than reflecting steady-state content health — worth a
   manual note in that period's review, not a silent re-rank.
2. **Feature distribution drift.** If `pct_impr_change` or `log_imp_prev` distributions
   shift meaningfully client-over-client (e.g. a new client type with very different
   traffic patterns joins the portfolio), the model's coefficients (fit on the current
   client mix) may no longer transfer — this is the same cold-start risk named in
   Section 2.
3. **Precision@20 tracked against real outcomes.** The honest next step beyond this
   notebook: have a reviewer mark, for a sample of past queue items, whether the
   flagged decline was real and actionable. If measured precision on that manual sample
   drops meaningfully from what Week 5/6 observed, that's a concrete retrain trigger.
4. **Retrain cadence.** Given the portfolio-level trend data in the FlyRank paper shows
   month-over-month growth, a practical starting cadence is **quarterly retrain**,
   with an out-of-cycle retrain triggered immediately by (1) or (2) above.
5. **Rule constants review.** `MIN_PREV_IMPR`, threshold values, and weights (`W_IMPR`
   etc.) were set in Week 4 and never re-derived from outcome data — worth revisiting
   once real reviewer feedback (point 3) exists.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
monitoring_snapshot = {
    "run_date": pd.Timestamp.now().strftime('%Y-%m-%d'),
    "base_rate_high_confidence_decline": float(d2['high_confidence_decline'].mean().round(4)),
    "n_scored_items": int(len(d)),
    "n_flagged_items": int(len(queue)),
    "n_eligible_clients": int(d['client_hash_id'].nunique()),
    "top20_single_client_share": float(client_share_top20.max()) if len(client_share_top20) else None,
    "suggested_retrain_cadence": "quarterly, or immediately on base-rate or feature-distribution drift",
}
print(json.dumps(monitoring_snapshot, indent=2))

{
  "run_date": "2026-08-11",
  "base_rate_high_confidence_decline": 0.5908,
  "n_scored_items": 99893,
  "n_flagged_items": 47332,
  "n_eligible_clients": 46,
  "top20_single_client_share": 0.45,
  "suggested_retrain_cadence": "quarterly, or immediately on base-rate or feature-distribution drift"
}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*



Write the ranked queue and a metrics/monitoring receipt to `work/outputs/` — these are
the exact files the capstone paper's Results and Ranked Recommendations sections build
on next week. Client/content IDs are hashed already (consistent with every prior week),
so this stays public-safe per the README.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
os.makedirs('work/outputs', exist_ok=True)

export_cols = [
    'rank', 'client_hash_id', 'content_hash_id',
    'action_score', 'model_score', 'model_confidence',
    'reason_codes_str', 'recommended_action',
    'imp_prev', 'imp_last', 'pct_impr_change',
    'pos_prev', 'pos_last', 'pos_change',
]
queue[export_cols].to_csv('work/outputs/w07_action_playbook_queue.csv', index=False)
print(f"wrote work/outputs/w07_action_playbook_queue.csv — {len(queue):,} rows")

with open('work/outputs/w07_monitoring_snapshot.json', 'w') as f:
    json.dump(monitoring_snapshot, f, indent=2)
print("wrote work/outputs/w07_monitoring_snapshot.json")

playbook_summary = {
    "no_go_actions": [
        "Auto-publishing or auto-editing content",
        "Auto-unpublishing/deindexing pages",
        "Using this score as an employee performance metric",
        "Client-facing reporting without human review",
        "Triggering budget/resourcing decisions directly off action_score",
    ],
    "human_review_checklist": [
        "Confirm decline isn't a known/communicated event",
        "For ACTIVITY_DROP: check indexing status directly before any content edit",
        "Sanity-check imp_prev against the reliability floor",
        "Cross-check model_confidence against reason_codes for disagreement cases",
    ],
    "retrain_triggers": [
        "Base-rate drift in high_confidence_decline",
        "Feature distribution drift (new client types, portfolio mix change)",
        "Measured precision@20 drop on a manual review sample",
        "Default cadence: quarterly",
    ],
}
with open('work/outputs/w07_playbook_summary.json', 'w') as f:
    json.dump(playbook_summary, f, indent=2)
print("wrote work/outputs/w07_playbook_summary.json")

wrote work/outputs/w07_action_playbook_queue.csv — 47,332 rows
wrote work/outputs/w07_monitoring_snapshot.json
wrote work/outputs/w07_playbook_summary.json


## Self-check

Before you submit, confirm each line honestly:

-  Every section above is filled — markdown thinking AND the code that backs it
-  The notebook runs top to bottom with no errors (Runtime → Run all)
-  No client names, URLs, or private queries anywhere
-  My claims use careful words: observed, measured, directional, decision-support
-  Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.